# Inside Modern LLMs: Architectures in 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week04_modern_llms.ipynb)

*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.*

**BME 6938 Advanced Biomedical AI · Week 4 · Reference notebook (ungraded)**

This notebook follows the Week 4 lecture and its four clips. It opens real, openly licensed models, looks inside them, and
reproduces every number from the clips. The running example continues Week 3: a decoder writing *"The patient denies chest pain"*.

| Section | Clip | What you will see |
|---|---|---|
| 1 | What changed since 2017 | A 2026 decoder block inside SmolLM2-360M: RMSNorm, SwiGLU, RoPE, grouped-query attention |
| 2 | The KV cache | The cache arithmetic from the clip, and the real cache measured inside SmolLM2-360M |
| 3 | Mixture of experts | A top-2 router written from scratch, then the real router of a mixture-of-experts model |
| 4 | More 2026 changes | Tokenizers old and new, stretching RoPE for long records, multi-token prediction |
| 5 | Optional | Gemma 3 and MedGemma, if you have a Hugging Face token |

**How to run it**
- **Google Colab:** *Runtime > Run all*. A GPU (*Runtime > Change runtime type > T4 GPU*) makes it faster, but CPU works.
- **HiPerGator or your own computer:** open it in Jupyter with a PyTorch kernel. It runs on CPU in a few minutes.
- The first run downloads two small models: SmolLM2-360M-Instruct (under 1 GB) and Granite 3.1 1B-A400M (about 2.7 GB).
  Both are Apache 2.0 licensed and need no account. For DeepSeek-V3 we download only its configuration file, not its 671 billion parameters.

**How to read it:** read the text above each cell, predict what the cell will print, then run it. Nothing here is graded.
**Credits.** The modern-block components in this notebook (RMSNorm, SwiGLU, rotary position embeddings, grouped-query attention, the KV cache, and the mixture-of-experts router) follow the approach of Sebastian Raschka's [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), which is licensed under Apache 2.0. The code here is rewritten for this course and its clinical examples.


## 0. Setup

In [ ]:
# Install the libraries this notebook needs if they are missing (Colab already has them).
import importlib, subprocess, sys
for package in ["torch", "transformers", "accelerate"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)                                            # fixed seed so random numbers are the same every run
np.set_printoptions(precision=2, suppress=True)
device = "cuda" if torch.cuda.is_available() else "cpu"         # use a GPU if there is one
dtype = torch.bfloat16 if device == "cuda" else torch.float32   # 16-bit on GPU, 32-bit on CPU (faster there)
print("PyTorch", torch.__version__, "| device:", device)

## 1. What changed since 2017  *(clip 1)*

SmolLM2-360M-Instruct is a small, recent model with every change from the clip. Its configuration lists the shape of the network.

In [ ]:
SMOL = "HuggingFaceTB/SmolLM2-360M-Instruct"
cfg = AutoConfig.from_pretrained(SMOL)                          # only the configuration file, a few kilobytes
head = cfg.hidden_size // cfg.num_attention_heads               # the config file does not store this
print("blocks (layers):     ", cfg.num_hidden_layers)          # 32, as in the clip
print("query heads:         ", cfg.num_attention_heads)        # 15
print("key/value heads:     ", cfg.num_key_value_heads)        # 5: grouped-query attention
print("head size:           ", head)                           # 64 numbers per head
print("feed-forward size:   ", cfg.hidden_size, "->", cfg.intermediate_size, "->", cfg.hidden_size)
print("activation:          ", cfg.hidden_act)                 # "silu", used inside SwiGLU
print("vocabulary entries:  ", cfg.vocab_size)

Now load the model itself and look at one block. Read the names: two RMSNorm layers (one before attention, one before the
feed-forward network), attention with separate query, key, and value projections, and an MLP with a gate projection.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(SMOL)
model = AutoModelForCausalLM.from_pretrained(SMOL, dtype=dtype).to(device).eval()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f} M parameters")
block = model.model.layers[0]                                   # the first of 32 decoder blocks
print(block)

### 1.1 RMSNorm: rescale by the root mean square
RMSNorm divides each vector by the square root of the mean of its squared entries, then multiplies by a learned weight.
Unlike LayerNorm, it does not subtract the mean. We write it in two lines and check it against the model's own layer.

In [ ]:
def rms_norm(x, weight, eps):
    # Divide by the root mean square of each vector, then scale by the learned weight.
    return weight * x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + eps)

x = torch.randn(5, cfg.hidden_size, device=device, dtype=dtype)   # 5 token vectors
ours = rms_norm(x.float(), block.input_layernorm.weight.float(), cfg.rms_norm_eps)
theirs = block.input_layernorm(x).float()
print("same as the model's RMSNorm:", torch.allclose(ours, theirs, atol=2e-2))

### 1.2 SwiGLU: a gated feed-forward network
The token is projected twice. One projection passes through SiLU and gates the other, element by element; a third matrix
projects the result back: `out = (SiLU(x·W_gate) × x·W_up)·W_down`.

In [ ]:
mlp = block.mlp                                                  # the model's feed-forward network
def swiglu(x):
    # Gate one projection with SiLU of the other, then project back down.
    gate = F.silu(mlp.gate_proj(x))                              # x·W_gate, through SiLU
    up = mlp.up_proj(x)                                          # x·W_up
    return mlp.down_proj(gate * up)                              # (gate × up)·W_down
with torch.no_grad():
    print("same as the model's MLP:", torch.allclose(swiglu(x).float(), mlp(x).float(), atol=2e-2))

### 1.3 RoPE and grouped-query attention
There is no position table in this model: RoPE rotates queries and keys inside attention (Week 2, clip 4). And the key and
value projections are a third as wide as the query projection, because 15 query heads share 5 key/value heads.

In [ ]:
att = block.self_attn
print("position table in the model:", any("position_embeddings" in n for n, _ in model.named_parameters()))
print("RoPE base (rope_theta):      ", getattr(cfg, "rope_theta", None) or cfg.rope_parameters["rope_theta"])
print("query projection:", tuple(att.q_proj.weight.shape), f"= {cfg.num_attention_heads} heads x {head}")
print("key projection:  ", tuple(att.k_proj.weight.shape), f"=  {cfg.num_key_value_heads} heads x {head}")
print("value projection:", tuple(att.v_proj.weight.shape), f"=  {cfg.num_key_value_heads} heads x {head}")

The overall shape is unchanged from 2017: a stack of blocks that predicts the next token. Let the model continue our note.

In [ ]:
prompt = "The patient denies chest pain"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=20, do_sample=False)   # greedy decoding (Week 3)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## 2. The KV cache and how models shrink it  *(clip 2)*

### 2.1 The arithmetic from the clip
Per token, the cache stores a key and a value for every layer and every key/value head:
`bytes per token = 2 × layers × key/value heads × head size × bytes per number`.

In [ ]:
def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_number=2):
    # Keys and values (2) for every layer and key/value head.
    return 2 * layers * kv_heads * head_dim * bytes_per_number

GB = 2 ** 30                                                     # one gigabyte (binary)
record = 32768                                                   # tokens in a long patient record
full = kv_bytes_per_token(32, 32, 128)                           # Llama 3 8B shape, every head with its own keys/values
gqa = kv_bytes_per_token(32, 8, 128)                             # the same model with 8 key/value heads
print(f"per token, 32 key/value heads: {full / 2**20:.2f} MB")
print(f"record, 32 key/value heads:    {full * record / GB:.0f} GB")
print(f"record, 8 key/value heads:     {gqa * record / GB:.0f} GB  ({full // gqa}x smaller)")

### 2.2 Latent attention in DeepSeek-V3
We download only DeepSeek-V3's configuration. Full attention would cache a key (128 + 64 numbers) and a value (128 numbers)
for each of 128 heads. Latent attention caches one compressed vector of 512 numbers plus one shared RoPE key of 64.

In [ ]:
ds = AutoConfig.from_pretrained("deepseek-ai/DeepSeek-V3")      # configuration only, no weights
full_mla = ds.num_attention_heads * ((ds.qk_nope_head_dim + ds.qk_rope_head_dim) + ds.v_head_dim)
latent = ds.kv_lora_rank + ds.qk_rope_head_dim
print("full keys and values per token and layer:", f"{full_mla:,}")
print("latent attention per token and layer:    ", latent)
print(f"about {full_mla / latent:.0f}x smaller")

### 2.3 The real cache inside SmolLM2-360M
`generate` keeps a cache object. We run the model once on our sentence and measure it: the formula should match.

In [ ]:
def cache_tensors(cache):
    # Return (key, value) tensors per layer for the cache formats of different transformers versions.
    if hasattr(cache, "layers"):
        return [(layer.keys, layer.values) for layer in cache.layers]
    if hasattr(cache, "key_cache"):
        return list(zip(cache.key_cache, cache.value_cache))
    return list(cache)

inputs = tokenizer("The patient denies chest pain", return_tensors="pt").to(device)
with torch.no_grad():
    cache = model(**inputs, use_cache=True).past_key_values     # the KV cache after reading the sentence
kv = cache_tensors(cache)
n_tokens = inputs.input_ids.shape[1]
print("layers in the cache:", len(kv), "| one key tensor (batch, kv heads, tokens, head size):", tuple(kv[0][0].shape))
measured = sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in kv)
predicted = kv_bytes_per_token(cfg.num_hidden_layers, cfg.num_key_value_heads, head, kv[0][0].element_size()) * n_tokens
print(f"measured {measured:,} bytes, formula {predicted:,} bytes, for {n_tokens} tokens")

### 2.4 Why cache at all?
Without the cache, every new token recomputes the keys and values of the whole note. We use a longer note (the sentence
repeated 40 times, about 250 tokens) and compare the time for 60 new tokens.

In [ ]:
import time
long_note = tokenizer(" ".join(["The patient denies chest pain."] * 40), return_tensors="pt").to(device)   # about 250 tokens
for use_cache in (True, False):
    start = time.time()
    with torch.no_grad():
        model.generate(**long_note, max_new_tokens=60, do_sample=False, use_cache=use_cache, min_new_tokens=60)
    print(f"use_cache={use_cache!s:5}: {time.time() - start:.1f} s")

### 2.5 Watch the cache grow, and watch what it saves

Section 2.1 did the arithmetic and section 2.4 said why the cache exists. The animation puts the two together as the
sequence gets longer.

The left panel is the cache itself: one column per token, filling as generation proceeds, with the memory it occupies
printed above. It grows in a straight line, which is the problem the whole of section 2.2 exists to attack.

The right panel is why we pay that price anyway. Without a cache, generating token `n` means attending over all `n`
positions again, so the total work grows with the square of the length. With a cache, each new token attends over stored
keys and values and the total grows linearly. The gap between the two curves is what the memory is buying.

In [ ]:
# An animation, drawn by the code below. It renders with to_jshtml, which needs no
# ffmpeg and plays in Colab, in Jupyter, and on nbviewer. Press play, or drag the
# slider to step through it frame by frame.
import numpy as np                                           # arrays
import matplotlib.pyplot as plt                              # figures
from matplotlib.animation import FuncAnimation               # the animation itself
from IPython.display import HTML                             # shows it in the notebook

UF, ORANGE, GREY = "#0021A5", "#FA4616", "#6B7280"           # the course colours
plt.rcParams["figure.dpi"] = 78                              # keeps the rendered animation light

N = 48                                                       # how many tokens we generate
KB_PER_TOKEN = 0.14                                          # megabytes of cache per token, a small-model figure
no_cache = np.cumsum(np.arange(1, N + 1))                    # work with no cache: 1 + 2 + ... + n, quadratic
with_cache = np.arange(1, N + 1)                             # work with a cache: one step per token, linear

fig, (left, right) = plt.subplots(1, 2, figsize=(9.4, 3.5),
                                  gridspec_kw={"width_ratios": [1.3, 1]})
fig.subplots_adjust(wspace=0.32)

def draw(i):
    n = i + 1                                                # tokens generated so far
    left.clear(); right.clear()

    # ---- left: the cache, one column per stored token
    left.set_xlim(0, N); left.set_ylim(0, 2.6); left.axis("off")
    left.bar(np.arange(n), np.ones(n), width=0.9, color=UF,
             edgecolor="white", linewidth=0.5)                   # the keys of each stored token
    left.bar(np.arange(n), np.ones(n), bottom=1.05, width=0.9, color="#5B8DEF",
             edgecolor="white", linewidth=0.5)                   # and the values
    left.text(0, 2.25, f"KV cache after {n} tokens: {n * KB_PER_TOKEN:.1f} MB",
              fontsize=11, color="#374151")
    left.text(-0.2, 0.5, "K", ha="right", va="center", fontsize=10, color=UF)
    left.text(-0.2, 1.55, "V", ha="right", va="center", fontsize=10, color="#5B8DEF")

    # ---- right: total attention work, with and without the cache
    right.plot(np.arange(1, n + 1), no_cache[:n], color=ORANGE, lw=2.2, label="recompute every step")
    right.plot(np.arange(1, n + 1), with_cache[:n], color=UF, lw=2.2, label="with a KV cache")
    right.set_xlim(1, N); right.set_ylim(0, no_cache[-1] * 1.05)
    right.set_xlabel("tokens generated", fontsize=10)
    right.set_ylabel("attention steps, cumulative", fontsize=10)
    right.legend(loc="upper left", fontsize=9, frameon=False)
    right.spines[["top", "right"]].set_visible(False)
    saved = no_cache[n - 1] - with_cache[n - 1]
    right.set_title(f"saved so far: {saved:,} steps", loc="right", fontsize=10, color=GREY)
    return []

ani = FuncAnimation(fig, draw, frames=N, interval=90)
plt.close(fig)
HTML(ani.to_jshtml(fps=12))

**Sliding windows** (Gemma 3, the base of MedGemma) are in the optional section 5, because Gemma needs a Hugging Face token.

## 3. Mixture of experts  *(clip 3)*

### 3.1 A top-2 router from scratch
We use the clip's illustrative router weights for *chest*: 16 experts, of which experts 3 and 11 score highest.

In [ ]:
weights = np.array([0.02, 0.04, 0.38, 0.03, 0.05, 0.02, 0.06, 0.03, 0.04, 0.02, 0.25, 0.01, 0.02, 0.01, 0.01, 0.01])
top2 = np.argsort(weights)[::-1][:2]                             # indices of the two largest weights
rescaled = weights[top2] / weights[top2].sum()                  # rescale so the chosen weights sum to 1
for e, w, r in zip(top2, weights[top2], rescaled):
    print(f"expert E{e + 1}: router weight {w:.2f} -> {r:.2f}")

A whole mixture-of-experts layer in a few lines: a linear router, top-k selection, and a weighted sum of the chosen experts.
Each expert is a small SwiGLU network, like the one in section 1.2.

In [ ]:
class Expert(nn.Module):
    # One small SwiGLU feed-forward network.
    def __init__(self, d, hidden):
        super().__init__()
        self.gate, self.up, self.down = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class MoE(nn.Module):
    # Route each token to its top-k experts and mix their outputs.
    def __init__(self, d=64, hidden=128, n_experts=16, k=2):
        super().__init__()
        self.router = nn.Linear(d, n_experts, bias=False)                      # one score per expert
        self.experts = nn.ModuleList(Expert(d, hidden) for _ in range(n_experts))
        self.k = k
    def forward(self, x):                                                       # x: (tokens, d)
        top_w, top_i = self.router(x).topk(self.k, dim=-1)                     # best k experts per token
        top_w = top_w.softmax(dim=-1)                                           # weights that sum to 1
        out = torch.zeros_like(x)
        for t in range(x.shape[0]):                                             # loop for clarity, not speed
            for w, e in zip(top_w[t], top_i[t]):
                out[t] += w * self.experts[int(e)](x[t])
        return out, top_i

moe = MoE()
tokens = ["denies", "chest", "pain"]
y, chosen = moe(torch.randn(3, 64))
for t, c in zip(tokens, chosen):
    print(f"{t:>7s} -> experts {[f'E{int(e) + 1}' for e in c]}")
active = sum(p.numel() for p in moe.router.parameters()) + 2 * sum(p.numel() for p in moe.experts[0].parameters())
print(f"parameters: {sum(p.numel() for p in moe.parameters()):,} in total, {active:,} used per token")

### 3.2 A real router: Granite 3.1 1B-A400M
This IBM model has 32 experts per layer and picks 8 for each token: 1.3 billion parameters in total, about 400 million
used per token. We record the router's input in the first layer and see which experts each token of our note chooses.

In [ ]:
GRANITE = "ibm-granite/granite-3.1-1b-a400m-instruct"
g_tok = AutoTokenizer.from_pretrained(GRANITE)
g_model = AutoModelForCausalLM.from_pretrained(GRANITE, dtype=dtype).to(device).eval()
g_cfg = g_model.config
print("experts per layer:", g_cfg.num_local_experts, "| chosen per token:", g_cfg.num_experts_per_tok)
print(f"{sum(p.numel() for p in g_model.parameters()) / 1e9:.2f} B parameters in total")

In [ ]:
router = g_model.model.layers[0].block_sparse_moe.router        # the router of the first layer
seen = {}
hook = router.register_forward_hook(lambda mod, inp, out: seen.update(x=inp[0].detach()))
ids = g_tok("The patient denies chest pain", return_tensors="pt").to(device)
with torch.no_grad():
    g_model(**ids)
hook.remove()

W = router.weight if hasattr(router, "weight") else router.layer.weight      # works across transformers versions
scores = (seen["x"].float() @ W.float().T).softmax(-1)                       # router weights for every expert
top_w, top_i = scores.topk(g_cfg.num_experts_per_tok, dim=-1)
for t, ws, es in zip([g_tok.decode([i]) for i in ids.input_ids[0]], top_w, top_i):
    print(f"{t:>10s} -> experts {sorted(int(e) + 1 for e in es)}")

Different tokens choose different experts, as in the clip. Granite's vocabulary is small (49,155 entries), so it splits
*denies* and *chest* into pieces; section 4.1 shows why larger vocabularies help. DeepSeek-V3 does the same at a much larger scale; its configuration
gives the numbers from the clip's last screen.

In [ ]:
print("routed experts per layer:", ds.n_routed_experts)
print("shared experts per layer:", ds.n_shared_experts)
print("chosen per token:        ", ds.num_experts_per_tok)
print("first dense layers:      ", ds.first_k_dense_replace, "(the rest use experts)")
print("total parameters: 671 B; used per token: 37 B (from the DeepSeek-V3 report)")

## 4. More 2026 changes  *(clip 4)*

### 4.1 Bigger vocabularies
The same clinical sentence through GPT-2's tokenizer (2019) and DeepSeek-V3's (2024). Only the tokenizers are downloaded.

In [ ]:
note = "Patient denies chest pain; troponin negative; ECG shows sinus tachycardia."
for name in ["gpt2", "deepseek-ai/DeepSeek-V3"]:
    tk = AutoTokenizer.from_pretrained(name)
    pieces = tk.tokenize(note)
    print(f"{name}: {len(tk):,} entries -> {len(pieces)} tokens")
    print("   ", [tk.convert_tokens_to_string([p]) for p in pieces])

### 4.2 Longer context: stretching the RoPE angles
Each RoPE pair rotates at its own speed. The slowest pair turns only a little during training on 4,096-token text; a position of
32,768 would rotate it 8 times further, to angles the model never saw. Rotating 8 times slower brings it back.

In [ ]:
d, base = 128, 10000.0                                           # head size and RoPE base, as in many models
slowest = base ** (-(d - 2) / d)                                 # speed of the slowest pair (radians per position)
for label, pos, stretch in [("trained, position 4,096", 4096, 1), ("position 32,768, no stretch", 32768, 1), ("position 32,768, 8x slower", 32768, 8)]:
    print(f"{label:30s}: slowest-pair angle = {pos * slowest / stretch:.2f} radians")
print()
print("DeepSeek-V3 RoPE scaling:", ds.rope_scaling if hasattr(ds, "rope_scaling") else ds.rope_parameters)
print("longest input after stretching:", f"{ds.max_position_embeddings:,}", "tokens")

DeepSeek-V3 uses YaRN, a refined version of this idea: it was pretrained on 4,096-token text, then stretched to 32K and to 128K
tokens with short extra training. Its configuration allows up to 4,096 × 40 = 163,840 positions.

### 4.3 Multi-token prediction
DeepSeek-V3 trains one extra module to predict the token after next. The configuration records how many.

In [ ]:
print("extra prediction modules (multi-token prediction):", ds.num_nextn_predict_layers)

## 5. Optional: Gemma 3 and MedGemma

Gemma and MedGemma are free but **gated**: accept the license on their Hugging Face pages
([Gemma 3](https://huggingface.co/google/gemma-3-1b-it), [MedGemma](https://huggingface.co/google/medgemma-4b-it)), create an
access token, and add it in Colab under *Secrets* as `HF_TOKEN` (or run `huggingface-cli login` on your own computer).
Without a token this section only prints these instructions.

In [ ]:
import os
token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata                            # Colab secrets, if we are on Colab
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass
try:
    from huggingface_hub import get_token
    token = token or get_token()                                 # a token saved by huggingface-cli login
except Exception:
    pass

if token is None:
    print("No Hugging Face token found: skipping Gemma 3 and MedGemma.")
else:
    gm = AutoConfig.from_pretrained("google/gemma-3-1b-it", token=token)
    kinds = getattr(gm, "layer_types", None)                     # "sliding_attention" or "full_attention" per layer
    print("Gemma 3 sliding window:", gm.sliding_window, "tokens")
    if kinds:
        print("first 12 layers:", ["local" if k == "sliding_attention" else "GLOBAL" for k in kinds[:12]])
    med = AutoConfig.from_pretrained("google/medgemma-4b-it", token=token)
    print("MedGemma 4B text model:", med.text_config.num_hidden_layers, "layers,",
          med.text_config.num_attention_heads, "query heads,", med.text_config.num_key_value_heads, "key/value heads")

## Recap

| Clip | Code in this notebook |
|---|---|
| 1. What changed since 2017 | `rms_norm`, `swiglu`, the SmolLM2-360M configuration and block |
| 2. The KV cache | `kv_bytes_per_token`, DeepSeek-V3's latent size, the measured cache, `use_cache` timing |
| 3. Mixture of experts | `MoE` from scratch, the Granite router, DeepSeek-V3's expert counts |
| 4. More 2026 changes | GPT-2 versus DeepSeek-V3 tokens, RoPE stretching, multi-token prediction |

> **Try it (optional):** in section 1, change the prompt to a short discharge summary opening and compare greedy decoding with
> sampling (`do_sample=True, temperature=0.7`). In section 3.2, which experts does "tachycardia" choose?